In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# On Colab the clone above puts `data/` beside the notebook, not one level up,
# so the path to the CSV has to be resolved rather than hard-coded.
DATA = '../data' if os.path.isdir('../data') else 'data'
# ---------------------------------------------------------------------------

# Cell potentials, equilibrium constants, and pH (Table 14.6-1 and Illustrations 14.6-1, 14.6-2 and 14.6-3)

$$\Delta_{\rm rxn}G^{\circ}=-nFE^{\circ}\qquad\qquad \ln K_a=\frac{nFE^{\circ}}{RT}$$

Section 14.6 is one equation used four ways. The half-cell potential is not new
thermodynamics -- it is $\Delta_{\rm rxn}G^{\circ}$ in volts -- and once that is accepted the
section's four questions are all the same question:

| | the question | what it adds |
|---|---|---|
| **Table 14.6-1** | what is $\Delta_{\rm rxn}G^{\circ}$ for sixteen half-reactions? | nothing but $-nFE^{\circ}$, applied sixteen times |
| **14.6-1** | the standard cell potential of a full cell | that half-cell potentials add, with a sign rule |
| **14.6-2** | the solubility product of AgCl from two half-cells | that $K_a$ for a reaction with no electrons in it can still come from electrode data |
| **14.6-3** | the voltage from a *concentration* difference alone | the activity coefficient, so Debye-Huckel from Sec. 9.10 |

Two of the three columns of Table 14.6-1 are **derived**, not measured, so this notebook
generates them from $E^{\circ}$ and $n$ rather than carrying them as data. The table in the
book is this notebook's output.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd

from thermo.electrolytes import DebyeHuckel

R = 8.314          # J/(mol K)
F = 9.6485e4       # C/mol
T = 298.15
RT = R * T
print(f"  RT   = {RT:.1f} J/mol")
print(f"  RT/F = {RT/F*1e3:.1f} mV        the book's 25.7 mV")
print(f"  F/RT = {F/RT:.4f} 1/V")

  RT   = 2478.8 J/mol
  RT/F = 25.7 mV        the book's 25.7 mV
  F/RT = 38.9238 1/V


## Table 14.6-1

Only $E^{\circ}$ and $n$ are data, and `code/data/half_cell_potentials.csv` ships those two
columns and nothing else. $\Delta_{\rm rxn}G^{\circ}/RT$ and $\Delta_{\rm rxn}G^{\circ}$ in kJ
both follow from $\Delta_{\rm rxn}G^{\circ} = -nFE^{\circ}$, so there is no reason to store
them -- and a stored derived column is a column that can drift from the data above it.

**Do not "correct" the $E^{\circ}$ column.** Every potential in it matches the CRC
Electrochemical Series to 0.01 V. Au$^+$/Au is the one soft entry -- Au$^+$ disproportionates,
so the potential is derived rather than measured, and compilations run from about 1.69 V to
about 1.83 V -- and the file carries the CRC value, 1.69, which is what the rest of the table
follows. The caveat is in the file's own header.

In [3]:
cells = pd.read_csv(f"{DATA}/half_cell_potentials.csv", comment="#")

cells["dG_kJ"] = -cells.electrons * F * cells.E0_V / 1e3
cells["dG_over_RT"] = cells.dG_kJ * 1e3 / RT
cells["ln_Ka"] = -cells.dG_over_RT

out = cells.assign(reaction=lambda d: d.oxidant + " + " + d.electrons.astype(str)
                   + "e- = " + d.reductant)
print(out[["reaction", "E0_V", "dG_over_RT", "dG_kJ", "ln_Ka"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

# Regression pins. These are this notebook's own numbers, not a comparison against
# anything -- they exist so that a change to the data file or to F, R or T cannot alter
# the printed table silently.
pin = out.set_index("reaction")
for rxn, dG, rt in (("Au+ + 1e- = Au",     -163.06,  -65.78),
                    ("Ag+ + 1e- = Ag",      -77.19,  -31.14),
                    ("Hg2 2+ + 2e- = 2Hg", -152.45,  -61.50),
                    ("Na+ + 1e- = Na",      261.47,  105.48)):
    assert abs(pin.loc[rxn, "dG_kJ"] - dG) < 0.05, rxn
    assert abs(pin.loc[rxn, "dG_over_RT"] - rt) < 0.01, rxn

# and the identity that makes the two derived columns one column
assert np.allclose(out.dG_over_RT, out.dG_kJ * 1e3 / RT)
assert np.allclose(out.ln_Ka, -out.dG_over_RT)
print(f"\n  {len(out)} rows, both derived columns generated from E0 and n")

             reaction  E0_V  dG_over_RT   dG_kJ   ln_Ka
       Au+ + 1e- = Au  1.69      -65.78 -163.06   65.78
     Cl2 + 2e- = 2Cl-  1.36     -105.87 -262.44  105.87
     Br2 + 2e- = 2Br-  1.09      -84.85 -210.34   84.85
       Ag+ + 1e- = Ag  0.80      -31.14  -77.19   31.14
   Hg2 2+ + 2e- = 2Hg  0.79      -61.50 -152.45   61.50
    Fe3+ + 1e- = Fe2+  0.77      -29.97  -74.29   29.97
      Cu2+ + 2e- = Cu  0.34      -26.47  -65.61   26.47
AgCl + 1e- = Ag + Cl-  0.22       -8.56  -21.23    8.56
       2H+ + 2e- = H2  0.00       -0.00   -0.00    0.00
      Fe3+ + 3e- = Fe -0.04        4.67   11.58   -4.67
      Pb2+ + 2e- = Pb -0.13       10.12   25.09  -10.12
      Zn2+ + 2e- = Zn -0.76       59.16  146.66  -59.16
      Al3+ + 3e- = Al -1.66      193.84  480.50 -193.84
      Mg2+ + 2e- = Mg -2.36      183.72  455.41 -183.72
       Na+ + 1e- = Na -2.71      105.48  261.47 -105.48
       Li+ + 1e- = Li -3.05      118.72  294.28 -118.72

  16 rows, both derived columns generated from 

## Illustration 14.6-1 -- half-cell potentials add

$$\mathrm{Cu^{2+}(aq)}+\mathrm{Zn(s)}\rightarrow\mathrm{Cu(s)}+\mathrm{Zn^{2+}(aq)}$$

The second half-reaction runs backwards relative to the table, so its potential changes sign. The
illustration then computes $\ln K_a$ twice -- once through $F$ and $RT$, once through the 25.7 mV
shortcut. Both are reproduced below, because the two routes round differently.

In [4]:
E_Cu = 0.34        # Cu2+ + 2e- -> Cu, as tabulated
E_Zn = -0.76       # Zn2+ + 2e- -> Zn, as tabulated; reversed below
n = 2
E_cell = E_Cu + (-E_Zn)
print(f"  E(cell) = {E_Cu:+.2f} + {-E_Zn:+.2f} = {E_cell:+.2f} V")

lnKa_full = n * F * E_cell / RT
lnKa_short = n * E_cell * 1e3 / 25.7
print(f"  ln Ka (via F and RT)   = {lnKa_full:.2f}")
print(f"  ln Ka (via 25.7 mV)    = {lnKa_short:.2f}")
print(f"  Ka = {np.exp(lnKa_full):.3g}")
print(f"\n  dG_rxn = {-n*F*E_cell/1e3:.1f} kJ/mol")
print(f"  the 25.7 mV shortcut costs {abs(lnKa_short - lnKa_full)/lnKa_full:.2%} in ln Ka,")
print(f"  which is a factor of {np.exp(abs(lnKa_short - lnKa_full)):.2f} in Ka itself")

assert abs(E_cell - 1.10) < 5e-3
assert abs(lnKa_full - 85.63) < 0.02 and abs(lnKa_short - 85.60) < 0.02

  E(cell) = +0.34 + +0.76 = +1.10 V
  ln Ka (via F and RT)   = 85.63
  ln Ka (via 25.7 mV)    = 85.60
  Ka = 1.55e+37

  dG_rxn = -212.3 kJ/mol
  the 25.7 mV shortcut costs 0.03% in ln Ka,
  which is a factor of 1.03 in Ka itself


## Illustration 14.6-2 -- a solubility product from electrode data

$$\mathrm{AgCl}\rightarrow\mathrm{Ag^+}+\mathrm{Cl^-}$$

No electrons appear in the overall reaction, and it is not a redox reaction at all -- yet it is the
difference of two half-reactions, so electrode data delivers its equilibrium constant. The check is
external: Illustration 13.2-3 gets the same $K_s$ from **solubility** measurements in potassium
nitrate solutions, by an entirely different route.

In [5]:
E_AgCl = 0.22      # AgCl + e- -> Ag + Cl-
E_Ag = 0.80        # Ag+  + e- -> Ag,  reversed below
E = E_AgCl - E_Ag
lnKa = 1 * F * E / RT
Ks = np.exp(lnKa)
print(f"  E = {E_AgCl:+.2f} - ({E_Ag:+.2f}) = {E:+.2f} V")
print(f"  ln Ka = {lnKa:.3f}")
print(f"  Ks    = {Ks:.3g}")

# The check that matters is not against this chapter -- it is against Chapter 13, which
# reaches the same Ks from solubility measurements by a completely different route.
Ks_ch13 = 1.607e-10          # Illustration 13.2-3, from solubility data
print(f"\n  Ks from Ill. 13.2-3 (solubility route) = {Ks_ch13:.4g}")
print(f"  the two routes differ by {abs(Ks-Ks_ch13)/Ks_ch13:.1%}, i.e. "
      f"{abs(lnKa - np.log(Ks_ch13))*RT/1e3:.2f} kJ/mol in dG")
print(f"  which is {abs(lnKa - np.log(Ks_ch13))*RT/F*1e3:.1f} mV -- less than the last digit of E0")

M = np.sqrt(Ks)
print(f"\n  saturation:  M(Ag+) = M(Cl-) = sqrt(Ks) = {M:.4g} M")
print(f"  as a mass:   {M * 143.32 * 1e3:.2f} mg AgCl per liter")

assert abs(lnKa - (-22.576)) < 0.01
assert abs(Ks - 1.57e-10) / 1.57e-10 < 0.01
assert abs(Ks - Ks_ch13) / Ks_ch13 < 0.03      # the two routes must stay within 3 %

  E = +0.22 - (+0.80) = -0.58 V
  ln Ka = -22.576
  Ks    = 1.57e-10

  Ks from Ill. 13.2-3 (solubility route) = 1.607e-10
  the two routes differ by 2.4%, i.e. 0.06 kJ/mol in dG
  which is 0.6 mV -- less than the last digit of E0

  saturation:  M(Ag+) = M(Cl-) = sqrt(Ks) = 1.252e-05 M
  as a mass:   1.79 mg AgCl per liter


**Two routes to the same number, and the gap is the useful part.** The electrode
route gives $K_s = 1.57\times10^{-10}$; Illustration 13.2-3 gets $1.607\times10^{-10}$ from
solubility measurements in potassium nitrate solutions. They differ by 2.4 %, which sounds like
a disagreement until it is converted into the units the measurement was made in: **0.6 mV**,
which is smaller than the last printed digit of $E^{\circ}$.

So the two routes agree to the precision of the data, and the comparison matters
precisely because it crosses chapters -- an equilibrium constant from a table of voltages, and
the same constant from dissolving a salt in water and measuring what dissolved.

## Illustration 14.6-3 -- voltage from a concentration difference

Two beakers of copper sulfate, 0.0001 M and 0.01 M, same electrodes. The standard potentials
cancel exactly, so **the entire voltage is an activity ratio** -- and at these concentrations the
activity coefficient is not one, which is where Sec. 9.10 comes back.

For a 2:2 electrolyte $|z_+z_-| = 4$ and $I = 4M$, so the Debye-Huckel limiting law gives
$\ln\gamma_{\pm} = -\alpha\,|z_+z_-|\sqrt{I} = -8\alpha\sqrt{M}$, which is where the illustration's
factor of 8 comes from.

In [6]:
M1, M2 = 1e-4, 1e-2
ALPHA_BOOK = 1.178          # the value Secs. 13.5 and 14.6 use
dh = DebyeHuckel("CuSO4")   # Table 9.10-1 route, for comparison
print(f"  alpha: {ALPHA_BOOK} (used by Secs. 13.5 and 14.6)   "
      f"vs {dh.alpha} (Table 9.10-1, as thermo carries it)")
print(f"  check: 0.5116 * ln 10 = {0.5116*np.log(10):.4f}  -- 1.178 is the natural-log partner")
print(f"  |z+ z-| = 4 and I = 4M, so the prefactor is 2 * 4 = 8\n")

def E_cell(alpha):
    ideal = RT / F * np.log(M1 / M2)
    activity = -RT / F * alpha * 8 * (np.sqrt(M1) - np.sqrt(M2))
    return ideal + activity, ideal, activity

for label, alpha in (("book, alpha = 1.178", ALPHA_BOOK),
                     ("Table 9.10-1, alpha = 1.175", dh.alpha),
                     ("ideal solution, alpha = 0", 0.0)):
    tot, ideal, act = E_cell(alpha)
    print(f"  {label:30s} -E = {tot*1e3:+7.2f} mV   "
          f"(ideal {ideal*1e3:+7.2f}, activity {act*1e3:+6.2f})")
tot, ideal, act = E_cell(ALPHA_BOOK)
print(f"\n  E = {abs(tot)*1e3:.1f} mV")
assert abs(abs(tot)*1e3 - 96.5) < 0.1        # regression pin on this cell's own value
print(f"  neglecting the activity coefficient would give {abs(ideal)*1e3:.1f} mV, "
      f"an error of {abs(act/tot):.1%}")

  alpha: 1.178 (used by Secs. 13.5 and 14.6)   vs 1.175 (Table 9.10-1, as thermo carries it)
  check: 0.5116 * ln 10 = 1.1780  -- 1.178 is the natural-log partner
  |z+ z-| = 4 and I = 4M, so the prefactor is 2 * 4 = 8

  book, alpha = 1.178            -E =  -96.52 mV   (ideal -118.31, activity +21.79)
  Table 9.10-1, alpha = 1.175    -E =  -96.58 mV   (ideal -118.31, activity +21.73)
  ideal solution, alpha = 0      -E = -118.31 mV   (ideal -118.31, activity  +0.00)

  E = 96.5 mV
  neglecting the activity coefficient would give 118.3 mV, an error of 22.6%


## The pH scale, Eq. 14.6-11 to 14.6-13

$$\mathrm{pH}=-\log a_{\mathrm{H^+}}\qquad\qquad
E=E^{\circ}-2.303\frac{RT}{F}\,\mathrm{pH}-\frac{RT}{2F}\ln\!\left[a_{\mathrm{H_2}}a^2_{\mathrm{M^+}}\right]$$

The coefficient $2.303\,RT/F$ is why a pH electrode reads what it reads: **one pH unit is 59.2 mV
at 25 °C, and nothing about that is adjustable.** Compute it rather than quote it: the
temperature dependence catches people out -- the slope of a pH electrode is a function
of temperature, which is why meters have a temperature probe.

In [7]:
slope = lambda T_: 2.303 * R * T_ / F * 1e3        # mV per pH unit
print(f"  d E / d pH at 25 C = {slope(298.15):.2f} mV per pH unit")
for T_ in (273.15, 298.15, 310.15, 373.15):
    print(f"    {T_-273.15:5.0f} C   {slope(T_):.2f} mV/pH")
print(f"\n  a meter calibrated at 25 C and used at 37 C misreads by "
      f"{(slope(310.15)/slope(298.15)-1):.1%} of the offset from the calibration pH,")
print(f"  i.e. {abs(slope(310.15)-slope(298.15))*3:.1f} mV -- "
      f"{abs(slope(310.15)-slope(298.15))*3/slope(310.15):.2f} pH units -- three units away from it")

# Eq. 14.6-11's last equality holds only when the activity coefficient is unity.
print("\n  Eq. 14.6-11 writes pH = -log a(H+) ~ -log[M(H+)/1 molal]. What that costs:")
for M in (1e-7, 1e-3, 1e-2, 1e-1):
    g = np.exp(DebyeHuckel("HCl", alpha=ALPHA_BOOK).ln_gamma_pm_from_I(M))
    print(f"    M = {M:.0e}   gamma_pm = {g:.4f}   "
          f"pH(activity) - pH(concentration) = {-np.log10(g):+.3f}")

  d E / d pH at 25 C = 59.17 mV per pH unit
        0 C   54.21 mV/pH
       25 C   59.17 mV/pH
       37 C   61.55 mV/pH
      100 C   74.05 mV/pH

  a meter calibrated at 25 C and used at 37 C misreads by 4.0% of the offset from the calibration pH,
  i.e. 7.1 mV -- 0.12 pH units -- three units away from it

  Eq. 14.6-11 writes pH = -log a(H+) ~ -log[M(H+)/1 molal]. What that costs:
    M = 1e-07   gamma_pm = 0.9996   pH(activity) - pH(concentration) = +0.000
    M = 1e-03   gamma_pm = 0.9634   pH(activity) - pH(concentration) = +0.016
    M = 1e-02   gamma_pm = 0.8889   pH(activity) - pH(concentration) = +0.051
    M = 1e-01   gamma_pm = 0.6890   pH(activity) - pH(concentration) = +0.162


## Your turn

1. Table 14.6-1's two derived columns are generated here from $E^{\circ}$ and $n$, so the table
   cannot disagree with itself. Show what that discipline prevents: perturb one $E^{\circ}$ in
   `half_cell_potentials.csv` by 0.01 V, regenerate, and say which printed numbers move and by
   how much. Then say what would have to go wrong for a *stored* $\Delta G$ column to drift from
   its own $E^{\circ}$ without anyone noticing.
2. Illustration 14.6-1 gets $\ln K_a = 85.6$, so $K_a \approx 10^{37}$. Compute the equilibrium
   copper-ion concentration in a cell that started at 0.1 M, and say how many copper atoms are
   left in a liter. Then say what that means for the usefulness of $K_a$ here, and what quantity
   an engineer would use instead.
3. The AgCl solubility product from electrode data sits 2.4 % below the value Illustration 13.2-3
   gets from solubility measurements. Convert that 2.4 % into millivolts and compare it with the
   precision of the $E^{\circ}$ column. Which of the two routes is the more precise, and would
   carrying $E^{\circ}$ to three decimals close the gap or just move it?
4. Illustration 14.6-3's factor of 8 comes from $|z_+z_-|\sqrt{I}$ for a 2:2 salt. Redo the
   calculation for the same concentrations of a 1:1 salt (KCl) and a 3:1 salt (AlCl$_3$). Rank the
   three by the size of the activity correction, and check the ranking against
   $|z_+z_-|\sqrt{|z_+z_-|}$.
5. A pH electrode's slope is $2.303\,RT/F$. Suppose a meter is calibrated in buffers at 25 °C and
   then used on blood at 37 °C. Work out the error in reported pH at pH 7.4, and say whether it
   matters clinically. Then explain why the calibration temperature, not the sample temperature, is
   the one that has to be recorded.
6. Sec. 14.6 says a single half-cell potential *"cannot be measured"* and the table is built by
   assigning zero to the hydrogen electrode. Show that every quantity computed in this notebook is
   invariant to that choice -- shift every $E^{\circ}$ in the table by the same constant and see
   what happens. Which quantities are invariant, and which one is not?